# Unix 3A: inputs, file products, and workflow planning

This lesson prepares the mapping workflow. It stops before full alignment, so students can focus on inputs, output formats, and the logic of the local analysis.

## Learning outcomes

By the end you should be able to verify the supplied data, explain each mapping product, and describe the order of a read-mapping workflow before running it locally.


## Lesson route

1. Confirm the working directory and input integrity.
2. Inspect FASTA and compressed FASTQ safely.
3. Connect FASTA and FASTQ files to the mapping question.
4. Order the files in the mapping workflow.
5. Explain which files are inputs, outputs, indexes, summaries, and temporary intermediates.


## Why this session starts with file literacy

Read mapping is not just a command to run. It is a chain of assumptions about input files, reference sequences, software choices, resources, and output formats. If one assumption is wrong, the final BAM or coverage table can still look plausible.

Before mapping, we therefore ask simple questions:

- Do the expected files exist?
- Are the compressed read files intact?
- What biological material does the reference contain?
- How many reads and bases are going into the analysis?
- What files should the workflow produce, and which are only temporary intermediates?

The aim of Unix 3A is to make the later mapping command interpretable rather than mysterious.


## 1. Working safely

Start Jupyter in the `UNIX_session_3` directory. The raw FASTQ files and reference FASTA are teaching inputs, so treat them as read-only. This notebook checks and inspects the inputs, but it does not create full alignment files.

A useful habit is to separate supplied data from generated work. In this session, supplied files live in `barcode09/` and `reference/`; files made during analysis should go under `work/` or `logs/`.

For now, focus on what the files contain and how the local command-line workflow fits together.


## 2. Input preflight

The preflight confirms that the FASTA exists, all five FASTQ files pass gzip integrity checks, and record totals match the teaching dataset.


### What the preflight protects against

Bioinformatics workflows often fail quietly. A command may run using the wrong folder, a partly transferred FASTQ file, or a reference with missing records. The preflight check is a small reproducibility step: it confirms that the practical starts from the intended dataset.

For compressed FASTQ files, integrity checking matters because the file may look present in `ls` even if a transfer was interrupted. For the reference, record and base counts help confirm that the expected chromosome and plasmids are present.


In [ ]:
%%bash
./preflight.sh --inputs-only


### Expected preflight result

- 5 reference records and 4,012,900 reference bases
- 5 compressed FASTQ files
- 16,801 reads and 60,751,725 read bases

If a count differs, stop before mapping and identify the missing, corrupt, or substituted input.


## 3. FASTA inspection

A **reference genome** is the sequence we align reads against. It provides the coordinate system for the analysis: after mapping, positions in the BAM and coverage table refer back to this FASTA file.

Reference genomes are commonly stored in **FASTA** format. Each record starts with a header line beginning with `>`, followed by one or more lines of sequence. A bacterial reference may contain a chromosome plus plasmids or other contigs. The supplied reference contains a chromosome and four plasmids.

Inspecting the headers tells us what reference sequences can appear later in SAM, BAM, `idxstats`, and coverage output.


In [ ]:
%%bash
grep '^>' reference/HVol_Complete.fasta
awk '{sub(/\r$/, "")} /^>/{if (name) print name, bases; name=$1; bases=0; next} {bases += length($0)} END{print name, bases}' reference/HVol_Complete.fasta


## 4. FASTQ inspection

A **FASTQ** file stores sequencing reads and their base-quality scores. Each read normally occupies four lines:

1. an identifier line beginning with `@`;
2. the nucleotide sequence;
3. a separator line beginning with `+`;
4. a quality string, with one quality character per base.

The quality string encodes how confident the sequencing instrument was about each base call. Higher-quality bases are more reliable. Many mappers can use this information during alignment or reporting.

The files here end in `.gz`, so they are compressed. Use streaming commands such as `gzip -cd` to inspect them without expanding large temporary files onto disk.


In [4]:
%%bash
gzip -cd barcode09/*_0.fastq.gz | awk 'NR <= 8 {print} NR == 8 {exit}'


@a464ee19-177e-4d99-bf42-9b65b7451c9e runid=1638da7d5a405764db8440d5cff8e77dcc6dbdb8 read=31 ch=615 start_time=2023-11-14T11:36:38.924627+00:00 flow_cell_id=PAH51710 protocol_group_id=IC_199 sample_id=Haloferax_10_barcodes barcode=barcode09 barcode_alias=barcode09 parent_read_id=a464ee19-177e-4d99-bf42-9b65b7451c9e basecall_model_version_id=dna_r9.4.1_e8_hac@v3.3
CAGTTCAGCCCTCGATGCAGATTGTTTAACCCATTAGGCACAGCGAGTCTTGGTTTGTTTTTCGCATTTATCGTGAAGCGGCTTTCGCGTTTTCGTGCGCCGCTTCAGACGTGTTCAACACGTCCACCGGCGAGAAGCCCTTCAGCGGCGACCCGCGTGGCGTCCTCAAGCGCGCTATCGAGCGCGCCGAGGAACTCGGCTATGACGTGAACGTCGCTTGAACCGGAGTTCTTCCTGTTCGAAGAGCGCGAAGACGGCCGCGCGACGACCGTCTGGGCGCCGGCGGCTACTTCGACTCGCCCCGAAGGACCTCGCGTCCGACGTGCGCCGCGACATCATCTACGGCCTCGAAAGCATGGGCTTCGACATCGAAGCCCTCGCACTGCAAGGTCGCCGAGGTCAACTACGAGATTAACTTCACGTACGACGACGCCCTCTCGACGGCCGACAACGTCGCAACGTTCCGGTCCGTCGTCCGCGCCATCGCGGCCGAACGGCCTCCGCGAGCTGATTCCGCCACCCGGCAATCCCGCGCATCAACGGCTCCGGCATGCGCACATCTCGCTGTTCAAGGACGGCGAGAACGCGTTCCACGACGGCAGCGACGAGTTCGACCTGAGCGACACGGCCAGAG

### Record check

For each record, the sequence and quality strings should have the same length. The four-line rule is safe for this conventional FASTQ dataset; more general parsers are preferable when format assumptions are uncertain.


### Why the sequence and quality lengths must match

In a valid conventional FASTQ record, the sequence line and quality line describe the same read, so they must have the same length. If they do not, the file may be truncated, malformed, or parsed incorrectly.

The check below is intentionally simple because this teaching dataset follows the standard four-line-per-record layout. For production workflows, specialist parsers are safer because FASTQ edge cases can be awkward.


In [ ]:
%%bash
gzip -cd barcode09/*_0.fastq.gz |
awk 'NR%4==2 {sequence_length=length($0)} NR%4==0 {if(length($0)!=sequence_length) bad++} END {print "records_checked", NR/4; print "length_mismatches", bad+0}'


## 5. Mapping products

Read mapping asks: **where does each read best fit in the reference?** For this practical we use Oxford Nanopore reads from `barcode09/` and align them to `reference/HVol_Complete.fasta`.

The conceptual workflow is:

```text
FASTQ reads + FASTA reference
        |
        v
Minimap2 alignment
        |
        v
SAM stream
        |
        v
Samtools sort
        |
        v
sorted BAM + BAM index
        |
        v
validation summaries and coverage tables
```

The practical streams SAM directly into `samtools sort`, so a large SAM file is never written. SAM is text alignment data, BAM is the compact binary version, and BAI is an index that lets tools access regions efficiently.


### Interactive workflow check

Edit the two variables in the next cell, then run it.

Your task is to:

1. put the workflow products in the order they appear in the mapping workflow;
2. label each item as one of `input`, `intermediate`, `primary result`, `index`, or `summary`.

This is deliberately about the *role* of each file or stream, not just its filename. For example, a SAM stream matters to the workflow, but we do not intend to keep it as a file.


After the checker passes, write one sentence in your notes explaining why the sorted BAM and depth table are retained, but the SAM stream is not written to disk.


## 6. From local commands to a reproducible workflow

Before reusing a command in a larger workflow, make sure the local workflow is clear. A good workflow records:

- the input reads and reference used;
- the command that converts reads into alignments;
- the sorted BAM and index that downstream tools need;
- validation summaries that show the result is structurally sound;
- coverage outputs and the denominator used to interpret them.

The next notebook runs this workflow locally. That local run is the learning version: you can inspect intermediate ideas, rerun commands, and check each result before moving on.


## 7. Planning the local run

The local workflow uses tools that can stream data from one command to another. The key design choice is that we do not keep a large SAM file. Instead, Minimap2 writes SAM to standard output and Samtools reads that stream directly:

```bash
minimap2 -ax map-ont reference/HVol_Complete.fasta barcode09/*.fastq.gz   | samtools sort -o work/aligned_barcode09.sorted.bam -
```

This pattern is useful because it reduces temporary files and makes the workflow easier to reproduce. In the next notebook, you will run the full version and then validate the result before interpreting coverage.


### Workflow planning activity

For each file or stream below, decide whether it is an input, primary result, index, summary, or disposable intermediate:

1. `barcode09/*.fastq.gz`
2. `reference/HVol_Complete.fasta`
3. SAM stream from Minimap2
4. sorted BAM
5. BAI index
6. `flagstat` text
7. depth TSV

Then explain why the sorted BAM and depth table should be retained, but the SAM stream does not need to be written to disk.


## Checkpoint before the next notebook...

You should now be able to answer:

- What is the difference between FASTA and FASTQ?
- Which command proves the compressed inputs are intact?
- Why does a FASTQ record need matching sequence and quality lengths?
- Why do we usually keep a sorted BAM rather than a large SAM file?
- Why is a sorted BAM indexed?
- Which outputs should be retained for interpretation?
- Which intermediate data can be streamed rather than saved?
